In [0]:
-- =====================================================================
-- CineClub Wrapped 2026 — Monthly members, savings only
-- Population: monthly1, Active / Active_Non_Renewing (yesterday's snapshot), ≥1 visit in 2026
-- =====================================================================

-- Population: active monthly members (yesterday's snapshot)
CREATE OR REPLACE TEMPORARY VIEW active_CC_Users AS (
    SELECT DISTINCT
        cc_user_profile_id,
        cde_id,
        plan_id,
        renewal_status
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE to_date(snapshot_date_id, 'yyyyMMdd') = curdate() - 1
      AND plan_id = 'monthly1'
      AND renewal_status IN ('Active', 'Active_Non_Renewing')
);

-- All 2026 transactions (visit date Jan 1 → yesterday), savings columns only
CREATE OR REPLACE TEMPORARY VIEW cc_all_trans AS (
    SELECT
        cc.cde_id,
        trans.visitdateid,
        trans.LocationId,
        trans.TicketFLAG,
        trans.ConcessionFLAG,
        trans.COTFLAG,
        trans.CINECLUBFLAG,
        trans.CineClub_Discount_Type_ID,
        trans.Quantity,
        trans.Net_Revenue
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS trans
    INNER JOIN active_CC_Users AS cc
        ON trans.cde_id = cc.cde_id
    WHERE trans.VisitDateId BETWEEN '20260101' AND date_format(curdate() - 1, 'yyyyMMdd')
);

-- Members with ≥1 visit (cde_id × date × location, refunds netted)
CREATE OR REPLACE TEMPORARY VIEW wrapped_members AS (
    SELECT DISTINCT cde_id
    FROM (
        SELECT cde_id, visitdateid, LocationId
        FROM cc_all_trans
        GROUP BY cde_id, visitdateid, LocationId
        HAVING SUM(Quantity) > 0
    ) AS v
);

-- Savings (owner savings logic, unchanged)
CREATE OR REPLACE TEMPORARY VIEW col_savings AS (
    WITH price_wide AS (
        SELECT * FROM VALUES
            ('1',   15.75, 16.00), ('2',   15.25, 15.50), ('3',   14.75, 15.00),
            ('4',   13.75, 14.00), ('5',   13.75, 14.00), ('6',   12.75, 13.00),
            ('8',   15.75, 16.00), ('9',   14.25, 14.50), ('11',  12.25, 12.50),
            ('12',  12.75, 13.00), ('14',  14.00, 14.25), ('15',  13.25, 13.50),
            ('16',  12.50, 12.75), ('17',  12.75, 13.00), ('18A', 13.75, 14.00),
            ('19',  11.25, 11.50), ('21',  15.25, 15.50), ('21J', 16.00, 16.25),
            ('22',  14.50, 14.75), ('23',  13.50, 13.75), ('24',  12.50, 12.75),
            ('24A', 14.50, 14.75), ('25',  12.25, 12.50), ('26',  13.75, 14.00),
            ('26A', 14.25, 14.50), ('27',  13.25, 13.50), ('28',  12.25, 12.50),
            ('29',  11.25, 11.50), ('30',  12.25, 12.50), ('31',  13.25, 13.50),
            ('32',  12.75, 13.00), ('33',  12.25, 12.50), ('34',  11.25, 11.50)
        AS t(Price_Category, price_oct25, price_nov26)
    ),
    price_category_dict AS (
        SELECT Price_Category, DATE'2025-10-01' AS Start_Date, DATE'2026-10-31' AS End_Date,
               CAST(price_oct25 AS DECIMAL(18,2)) AS Adult_General
        FROM price_wide
        UNION ALL
        SELECT Price_Category, DATE'2026-11-01', DATE'9999-12-31',
               CAST(price_nov26 AS DECIMAL(18,2))
        FROM price_wide
    ),
    price_hist AS (
        SELECT l.locationid AS LocationId, d.Adult_General, d.Start_Date, d.End_Date
        FROM data_exploration_hub.marketing_hub.mkt_pricing_tier_by_loc AS l
        JOIN price_category_dict AS d
          ON UPPER(TRIM(l.pricing_tier)) = d.Price_Category
    ),
    txn AS (
        SELECT
            cde_id, visitdateid, LocationId, Net_Revenue, Quantity,
            TicketFLAG, ConcessionFLAG, COTFLAG, CineClub_Discount_Type_ID,
            to_date(CAST(visitdateid AS STRING), 'yyyyMMdd') AS visit_date
        FROM cc_all_trans
        WHERE CINECLUBFLAG = 'Yes'
    ),
    price_lookup AS (
        SELECT d.LocationId, d.visit_date, p.Adult_General
        FROM (SELECT DISTINCT LocationId, visit_date FROM txn) AS d
        JOIN price_hist AS p
          ON  d.LocationId = p.LocationId
          AND d.visit_date BETWEEN p.Start_Date AND p.End_Date
        QUALIFY ROW_NUMBER() OVER (PARTITION BY d.LocationId, d.visit_date ORDER BY p.Start_Date DESC) = 1
    ),
    member_savings AS (
        SELECT
            t.cde_id,
            ROUND(SUM(CASE WHEN t.CineClub_Discount_Type_ID IN (5, 6)
                           THEN t.Quantity * (COALESCE(pl.Adult_General, 9.99) - 9.99) ELSE 0 END), 2)   AS ticket_savings_999,
            ROUND(SUM(CASE WHEN t.CineClub_Discount_Type_ID IN (7, 8)
                           THEN t.Quantity * (COALESCE(pl.Adult_General, 10.99) - 10.99) ELSE 0 END), 2) AS ticket_savings_1099,
            ROUND(SUM(CASE WHEN t.ConcessionFLAG = 'Yes' AND t.CineClub_Discount_Type_ID = 2
                           THEN t.Net_Revenue * 0.25 ELSE 0 END), 2)                                      AS concession_savings,
            ROUND(SUM(CASE WHEN t.ConcessionFLAG = 'Yes' AND t.CineClub_Discount_Type_ID = 3
                           THEN t.Net_Revenue * 0.25 ELSE 0 END), 2)                                      AS gaming_savings,
            ROUND(SUM(CASE WHEN t.TicketFLAG = 'Yes' AND t.COTFLAG = 'Yes'
                            AND t.CineClub_Discount_Type_ID IN (5, 6, 7, 8)
                           THEN t.Quantity * 1.50 ELSE 0 END), 2)                                         AS online_booking_savings
        FROM txn AS t
        LEFT JOIN price_lookup AS pl
          ON t.LocationId = pl.LocationId AND t.visit_date = pl.visit_date
        GROUP BY t.cde_id
    )
    SELECT
        cde_id,
        ticket_savings_999,
        ticket_savings_1099,
        concession_savings,
        gaming_savings,
        online_booking_savings,
        GREATEST(ROUND(ticket_savings_999 + ticket_savings_1099 + concession_savings
                       + gaming_savings + online_booking_savings, 2), 0) AS total_savings
    FROM member_savings
);

-- =====================================================================
-- Final output: one row per member
-- =====================================================================
CREATE OR REPLACE TEMPORARY VIEW wrapped_output AS (
    SELECT
        w.cde_id,
        a.plan_id,
        a.renewal_status,
        COALESCE(s.ticket_savings_999, 0)     AS ticket_savings_999,
        COALESCE(s.ticket_savings_1099, 0)    AS ticket_savings_1099,
        COALESCE(s.concession_savings, 0)     AS concession_savings,
        COALESCE(s.gaming_savings, 0)         AS gaming_savings,
        COALESCE(s.online_booking_savings, 0) AS online_booking_savings,
        COALESCE(s.total_savings, 0)          AS total_savings
    FROM wrapped_members AS w
    INNER JOIN active_CC_Users AS a ON w.cde_id = a.cde_id
    LEFT JOIN col_savings      AS s ON w.cde_id = s.cde_id
);

SELECT * FROM wrapped_output;